# TD 1 : faire parler la base de DATAFLOW en SQL

> **De :** Claire, directrice marketing, DATAFLOW. **À :** l'équipe data de SEP Conseil.
>
> Voici l'accès à notre base, `dataflow.db`. Avant de parler de modèle, j'aimerais comprendre ce qu'on a :
> combien de clients partent, lesquels, et si nos données sont fiables. Je présente un premier point à mon directeur vendredi.

**Linh :** « Claire veut des chiffres simples et justes. Pas de modèle aujourd'hui : des requêtes propres, et un œil critique sur les données. »

## Objectifs

À la fin de ce TD, vous saurez :
- interroger une base SQLite depuis Python, avec `sqlite3` et `pd.read_sql` ;
- utiliser `GROUP BY`, `JOIN`, `HAVING`, `CASE WHEN`, `WITH` et une fonction de fenêtre ;
- repérer dans les données des doublons, des manquants et des valeurs impossibles.

## Mode d'emploi

- **[SQL imposé]** (questions 1 à 4) : la question se traite en SQL. **[au choix]** : SQL ou pandas. **[fourni]** : la requête vous est donnée, vous la lisez et l'exécutez.
- Avant chaque requête, écrivez votre **prédiction** dans la cellule prévue : un chiffre, un ordre de grandeur, un classement. Puis exécutez et comparez.
- Une surprise n'est pas un échec : c'est une information.
- En équipe : une personne écrit la requête, une autre la relit à voix haute avant de l'exécuter. Changez de rôle à chaque question.

## Étape 0 : se connecter et regarder les tables

Exécutez les deux cellules suivantes sans rien modifier.

In [2]:
import sqlite3

import pandas as pd

con = sqlite3.connect("../../donnees/dataflow.db")


def sql(requete):
    """Exécute une requête sur la base DATAFLOW et renvoie un DataFrame."""
    return pd.read_sql(requete, con)


sql("SELECT name AS table_name FROM sqlite_master WHERE type = 'table'")

,table_name
0,clients
1,contrats
2,services
3,factures


In [3]:
for table in ["clients", "contrats", "services", "factures"]:
    print(f"--- {table} ---")
    print(sql(f"SELECT * FROM {table} LIMIT 3").to_string(index=False), "\n")

--- clients ---
customer_id gender  senior partner dependents
 7590-VHVEG Female       0     Yes         No
 5575-GNVDE   Male       0      No         No
 3668-QPYBK   Male       0      No         No 

--- contrats ---
customer_id       contract  tenure paperless   payment_method  churn  nb_appels_service_resiliation
 7590-VHVEG Month-to-month       1       Yes Electronic check      0                              0
 5575-GNVDE       One year      34        No     Mailed check      0                              0
 3668-QPYBK Month-to-month       2       Yes     Mailed check      1                              1 

--- services ---
customer_id          service valeur
 0002-ORFBO DeviceProtection     No
 0002-ORFBO  InternetService    DSL
 0002-ORFBO    MultipleLines     No 

--- factures ---
customer_id  monthly_charges  total_charges
 0236-HFWSV            93.35        1444.65
 2220-IAHLS            19.40          19.40
 0604-THJFP            75.95        4542.35 



**Première question à se poser :** pour chaque table, que représente **une ligne** ? Notez-le ici en une phrase par table.

- `clients` : un client unqiue avec son id comme clé primaire et ses caractéristiques
- `contrats` :
- `services` :
- `factures` :

## Question 1. Combien de clients, et quel taux de churn ? [SQL imposé]

Notions : `COUNT(*)`, `AVG`. Table : `contrats`. Rappel : `churn` vaut 1 si le client est parti, 0 sinon. La moyenne d'une colonne 0/1 est donc un taux.

**Votre prédiction :** *double-cliquez ici et écrivez votre réponse avant d'exécuter.*

In [7]:
requete = """
select count(distinct(customer_id)) as nb_clients, avg(churn) as churn_moyen
from contrats;
"""
sql(requete)

,nb_clients,churn_moyen
0,7043,0.26537


**Pour Claire :** écrivez le résultat en une phrase qu'elle peut répéter à son directeur.

## Question 2. Quel est le taux de churn par type de contrat ? [SQL imposé]

Notion : `GROUP BY`. Triez du plus fort au plus faible taux.

**Votre prédiction :** *quel contrat part le plus ? Environ combien de fois plus que le moins risqué ?*

In [18]:
requete = """
SELECT contract, avg(churn)
FROM contrats
GROUP BY contract;
"""
sql(requete)

,contract,avg(churn)
0,Month-to-month,0.427097
1,One year,0.112695
2,Two year,0.028319


**Pour Claire :** que peut-elle faire de ce résultat, dès lundi, sans aucun modèle ?

## Question 3. Les seniors partent-ils plus ? [SQL imposé]

Notion : `JOIN`. La colonne `senior` est dans `clients`, la colonne `churn` dans `contrats`. La clé commune est `customer_id`.

**Votre prédiction :** *plus, moins ou autant que les autres ?*

In [10]:
requete = """
select cl.senior, avg(co.churn)
from contrats as co 
join clients as cl on co.customer_id = cl.customer_id
group by cl.senior;
"""
sql(requete)

,senior,avg(co.churn)
0,0,0.236062
1,1,0.416813


**Question d'interprétation :** les seniors sont 1 142. Faut-il en faire la cible prioritaire de Claire ? Qu'aimeriez-vous vérifier avant ?

## Question 4. Combien de factures sont manquantes ou en double ? [SQL imposé]

Notions : `IS NULL`, `GROUP BY ... HAVING COUNT(*) > 1`. Écrivez deux requêtes :
1. le nombre de factures dont `total_charges` est manquant ;
2. le nombre de clients qui ont plus d'une facture.

**Votre prédiction :** *zéro, quelques-unes, des centaines ?*

In [ ]:
requete_manquants = """
select count(*) as nb_manquants
from factures
where total_charges IS NULL;
"""
sql(requete_manquants)

,nb_manquants
0,82


In [12]:
requete_doublons = """
select count(*) as nb_clients_en_double
from (
    select customer_id
    from factures
    group by  customer_id
    having count(*) > 1
);"""
sql(requete_doublons)

,nb_clients_en_double
0,140


**Question d'interprétation :** les doublons sont-ils des lignes **identiques**, ou deux factures différentes pour le même client ? Écrivez une requête pour le vérifier. Pourquoi est-ce important avant une jointure ?

In [13]:
requete = """
select customer_id, monthly_charges, total_charges, count(*) as nb_lignes
from factures
group by customer_id, monthly_charges, total_charges
having count(*) > 1;
"""
sql(requete)

,customer_id,monthly_charges,total_charges,nb_lignes
0,0014-BMAQU,84.65,5377.80,2
1,0067-DKWBL,49.25,91.10,2
2,0068-FIGTF,78.20,2078.95,2
3,0222-CNVPT,48.80,2555.05,2
4,0224-NIJLP,20.15,165.50,2
...,...,...,...,...
135,9746-MDMBK,113.45,5317.80,2
136,9850-OWRHQ,74.10,228.00,2
137,9851-QXEEQ,104.70,4346.40,2
138,9938-ZREHM,72.10,2658.40,2


## Question 5. Quels sont les 10 clients qui rapportent le plus ? [au choix]

Notion : `ORDER BY ... DESC LIMIT`. Utilisez `total_charges` (le total facturé depuis l'arrivée du client). Affichez aussi `tenure` (en mois) et `monthly_charges`.

**Votre prédiction :** *le plus gros total, environ combien ? Aidez-vous d'un calcul : facture mensuelle maximale fois ancienneté maximale.*


In [20]:
requete = """
select distinct(f.customer_id), f.total_charges, f.monthly_charges, c.tenure
from factures as f
join contrats as c on f.customer_id = c.customer_id
order by total_charges desc
limit 10;
"""
sql(requete)

,customer_id,total_charges,monthly_charges,tenure
0,1498-NHTLT,518600.0,86.75,59
1,4521-YEEHE,159475.0,88.85,18
2,5442-BXVND,83850.0,24.25,33
3,8402-OOOHJ,83515.0,20.65,41
4,1196-AMORA,52000.0,73.60,7
5,0219-YTZUE,37190.0,84.80,4
6,2023-VQFDL,34880.0,19.00,18
7,2266-FUBDZ,27885.0,19.35,15
8,5804-LEPIM,13905.0,70.35,2
9,2889-FPWRM,8684.8,117.80,72


**Question d'interprétation :** pour chaque ligne, calculez de tête `tenure` fois `monthly_charges`. Que remarquez-vous ? Que diriez-vous à Claire avant de lui envoyer ce classement ?

## Question 6. Combien de services a chaque client ? [au choix]

Notion : `COUNT` et `SUM` sur une table longue. La table `services` a une ligne par client **et** par service.
Commencez par regarder les valeurs possibles de la colonne `valeur` (cellule ci-dessous), puis comptez, pour chaque client, les services **vraiment souscrits**.
Affichez ensuite combien de clients ont 1 service, 2 services, etc.

In [6]:
sql("SELECT service, valeur, COUNT(*) AS nb FROM services GROUP BY service, valeur")

,service,valeur,nb
0,DeviceProtection,No,3095
1,DeviceProtection,No internet service,1526
2,DeviceProtection,Yes,2422
3,InternetService,DSL,2421
4,InternetService,Fiber optic,3096
5,InternetService,No,1526
6,MultipleLines,No,3390
7,MultipleLines,No phone service,682
8,MultipleLines,Yes,2971
9,OnlineBackup,No,3088


**Votre prédiction :** *combien de services a un client typique ?*

In [ ]:
requete = """
select customer_id, count(*) as nb_services
from services
where valeur not in ('No', 'No internet service', 'No phone service')
group by customer_id;
"""
sql(requete)

,customer_id,nb_services
0,0002-ORFBO,5
1,0003-MKNFE,4
2,0004-TLHLJ,3
3,0011-IGKFF,6
4,0013-EXCHZ,4
...,...,...
7038,9987-LUTYD,4
7039,9992-RRAMN,4
7040,9992-UJOEL,3
7041,9993-LHIEB,6


**Question d'interprétation :** Mohamed a écrit `SELECT customer_id, COUNT(*) FROM services GROUP BY customer_id` et trouve 9 services pour tout le monde. Où est son erreur ?

## Question 7. Quels moyens de paiement ont un churn supérieur à 30 % ? [au choix]

Notion : `HAVING`. Affichez aussi le nombre de clients de chaque moyen de paiement.

**Votre prédiction :** *lequel, ou lesquels ?*

In [27]:
requete = """
select payment_method, count(*) as nb_clients, avg(churn) as taux_churn
from contrats
group by payment_method
having avg(churn) > 0.3;
"""
sql(requete)

,payment_method,nb_clients,taux_churn
0,Electronic check,2365,0.452854


**Question d'interprétation :** pourquoi ne peut-on pas écrire cette condition dans un `WHERE` ? Et Claire doit-elle interdire ce moyen de paiement ?

## Question 8. Quel est le churn par tranche d'ancienneté ? [les deux]

Tranches : moins de 12 mois, de 12 à 24 mois, 24 mois et plus.
Écrivez **deux** versions : une en SQL avec `CASE WHEN`, une en pandas avec `pd.cut`. Puis comparez leur lisibilité.

**Votre prédiction :** *le churn monte ou descend avec l'ancienneté ? De combien ?*

In [31]:
requete = """
select case
           when tenure < 12 then '1. moins de 12 mois'
           when tenure < 24 then '2. de 12 à 24 mois'
           when tenure >= 24 then '3. 24 mois et plus'
       end as tranche,
       count(*) as nb_clients,
       avg(churn) as taux_churn
from contrats
group by tranche
order by tranche;
"""
sql(requete)

,tranche,nb_clients,taux_churn
0,1. moins de 12 mois,2069,0.482842
1,2. de 12 à 24 mois,1047,0.295129
2,3. 24 mois et plus,3927,0.142857


In [ ]:
contrats = sql("SELECT tenure, churn FROM contrats")
# À compléter
...

**Question d'interprétation :** quelle version préférez-vous, et pourquoi ? À quel moment d'un projet choisiriez-vous l'autre ?

## Question 9. Quels sont les 3 clients les plus chers de chaque type de contrat ? [au choix]

Notions : `RANK() OVER (PARTITION BY ... ORDER BY ...)` et `WITH`. Classez les clients par `monthly_charges`, du plus cher au moins cher, **à l'intérieur** de chaque type de contrat. Affichez aussi `churn`.

Deux indices :
- une fonction de fenêtre ne peut pas être filtrée dans le `WHERE` de la même requête : calculez le rang dans une étape `WITH`, puis filtrez ;
- pensez aux doublons de la question 4.

**Votre prédiction :** *ces clients très chers sont-ils plutôt fidèles ou plutôt partants ?*

In [ ]:
requete = """
-- À compléter
"""
sql(requete)

**Question d'interprétation :** quelle différence entre ce résultat et un `GROUP BY contract` ? Pourquoi Claire pourrait-elle vouloir appeler ces clients en priorité ?

## Question 10 — la table d'analyse du modèle **[fourni]**

C'est la requête la plus longue du TD, et vous n'avez pas à l'écrire. Lisez-la,
exécutez-la, et répondez à une seule question : **une ligne par client ?**
Elle resservira telle quelle au TD 3.

**Linh :** « Demain, on entraîne un modèle. Il lui faut **une table, une ligne par client**, avec tout ce qu'on sait de lui et la cible. C'est la requête la plus importante du projet. »

Colonnes attendues :

| Colonne | Table d'origine |
|---|---|
| `customer_id`, `gender`, `senior`, `partner`, `dependents` | `clients` |
| `contract`, `tenure`, `paperless`, `payment_method` | `contrats` |
| `monthly_charges`, `total_charges` | `factures`, **sans les doublons** |
| `nb_services` | `services` : le nombre de services souscrits (question 6) |
| `internet` | `services` : la valeur du service `InternetService` (DSL, Fiber optic ou No) |
| `churn` | `contrats` : la cible |

Notions : plusieurs étapes `WITH`, plusieurs jointures. Astuce pour `internet` : `MAX(CASE WHEN service = 'InternetService' THEN valeur END)`.

**Votre prédiction :** *combien de lignes doit avoir la table ? Et combien de colonnes ?*

In [ ]:
requete_features = """
WITH factures_propres AS (
    SELECT DISTINCT customer_id, monthly_charges, total_charges
    FROM factures
),
resume_services AS (
    SELECT customer_id,
           SUM(CASE WHEN valeur NOT IN ('No', 'No internet service', 'No phone service')
                    THEN 1 ELSE 0 END) AS nb_services,
           MAX(CASE WHEN service = 'InternetService' THEN valeur END) AS internet
    FROM services
    GROUP BY customer_id
)
SELECT cl.customer_id, cl.gender, cl.senior, cl.partner, cl.dependents,
       co.contract, co.tenure, co.paperless, co.payment_method,
       f.monthly_charges, f.total_charges,
       s.nb_services, s.internet,
       co.churn
FROM clients AS cl
JOIN contrats AS co              ON co.customer_id = cl.customer_id
LEFT JOIN factures_propres AS f  ON f.customer_id  = cl.customer_id
LEFT JOIN resume_services AS s   ON s.customer_id  = cl.customer_id
"""
table = sql(requete_features)
table.head()

**Vérifiez votre table** avec la cellule suivante. Si une ligne affiche `False`, votre requête a un problème : trouvez lequel.

In [ ]:
print("7 043 lignes :", len(table) == 7043)
print("une ligne par client :", table["customer_id"].is_unique)
print("14 colonnes :", table.shape[1] == 14)
print("taux de churn inchangé (26,5 %) :", round(table["churn"].mean(), 3) == 0.265)
print("manquants par colonne :")
print(table.isna().sum()[table.isna().sum() > 0])

**Question d'interprétation :** pourquoi un `LEFT JOIN` pour les factures, plutôt qu'un `JOIN` ? Que reste-t-il à nettoyer dans cette table avant d'entraîner un modèle ?

## Bilan : votre message à Claire

**Linh :** « Claire présente vendredi. Écris-lui 5 lignes au maximum : ce qu'on sait, ce qui nous surprend, ce qu'on fait ensuite. Pas de jargon. »

*Écrivez votre message ici.*